### Modelado gasto viajeros

Objetivo:
    - Cargar la base de viajeros 2023.
    - Construir la variable objetivo Gasto_Total_COP consolidando distintas monedas.
    - Generar un dataset filtrado y listo para la fase de modelado.
    


Este notebook debe ejecutarse desde la raíz del repositorio:
    ProyectoMIAD-DespliegueSoluciones/

#### Cargue de Datos

En esta celda cargamos:
    - La base principal de viajeros 2023.
    - El diccionario de datos asociado.
Las rutas son relativas a la estructura del repositorio.

In [3]:
import pandas as pd

ruta_datos = "data/viajeros_2023_datos_abiertos_final.xlsx"
ruta_diccionario = "data/diccionario-de-datos_viajeros_2023-da.xlsx"

df = pd.read_excel(ruta_datos)
dic = pd.read_excel(ruta_diccionario)

print(df.shape)
print(dic.shape)


(74547, 134)
(134, 3)


1. Normalizamos los nombres de las columnas para evitar problemas por espacios en blanco u otros caracteres extraños.

2. Construcción de la variable objetivo: Gasto_Total_COP

La encuesta reporta el gasto total en tres posibles columnas:
    - P_105A: Gasto en pesos colombianos (COP).
    - P_105B: Gasto en dólares (USD).
    - P_105C: Gasto en euros (EUR).

Regla de negocio:
    1. Si hay valor positivo en P_105A -> usar ese valor directamente.
    2. Si no, pero hay valor positivo en P_105B -> convertir a COP con tasa fija.
    3. Si no, pero hay valor positivo en P_105C -> convertir a COP con tasa fija.
    4. Si no hay información válida -> NaN.

Las tasas usadas son de referencia para este proyecto (ajustables si el equipo lo define):

In [4]:
import numpy as np

# Normalizar nombres de columnas (por si tienen espacios o inconsistencias)
df.columns = df.columns.str.strip()

# Definir tasas de conversión
USD_TO_COP = 4000
EUR_TO_COP = 4300

# Crear la columna objetivo consolidando las tres posibles monedas
def calcular_gasto_cop(row):
    if not pd.isna(row.get("P_105A")) and row["P_105A"] > 0:
        return row["P_105A"]
    elif not pd.isna(row.get("P_105B")) and row["P_105B"] > 0:
        return row["P_105B"] * USD_TO_COP
    elif not pd.isna(row.get("P_105C")) and row["P_105C"] > 0:
        return row["P_105C"] * EUR_TO_COP
    else:
        return np.nan

df["Gasto_Total_COP"] = df.apply(calcular_gasto_cop, axis=1)

df[["P_105A", "P_105B", "P_105C", "Gasto_Total_COP"]].head()


,P_105A,P_105B,P_105C,Gasto_Total_COP
0,3000000.0,NaN,NaN,3000000.0
1,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN


Revisamos Proporción de valores NO nulos en Gasto_Total_COP.


In [5]:
# Cuántos valores no nulos quedaron
df["Gasto_Total_COP"].isna().mean()


0.9136115470776826

Estadísticos descriptivos básicos.

In [6]:
df["Gasto_Total_COP"].describe().apply(lambda x: round(x, 2))


count    6.440000e+03
mean     2.617884e+06
std      7.329997e+06
min      1.000000e+00
25%      3.000000e+05
50%      8.000000e+05
75%      2.000000e+06
max      2.000000e+08
Name: Gasto_Total_COP, dtype: float64

 Creamos df_modelo: 
 - Solo filas con Gasto_Total_COP no nulo.
 - Solo valores positivos (descartamos errores o registros inconsistentes).

In [7]:
df_modelo = df.copy()
df_modelo = df_modelo[df_modelo["Gasto_Total_COP"].notna()]
df_modelo = df_modelo[df_modelo["Gasto_Total_COP"] > 0]

print(f"Dataset original: {df.shape}")
print(f"Dataset filtrado: {df_modelo.shape}")


Dataset original: (74547, 135)
Dataset filtrado: (6440, 135)


Guardamos el dataset procesado que será el insumo estándar para:
   - Entrenamiento de modelos.
   - Consumo desde scripts.
   - Posible uso en el dashboard si se requiere.

Archivo generado: data/processed/viajeros_2023_gasto_cop.csv

In [8]:
import os

os.makedirs("data/processed", exist_ok=True)
ruta_salida = "data/processed/viajeros_2023_gasto_cop.csv"
df_modelo.to_csv(ruta_salida, index=False, encoding="utf-8-sig")

print(f"Archivo guardado en: {ruta_salida}")


Archivo guardado en: data/processed/viajeros_2023_gasto_cop.csv
